# Candidate models

The record model is deliberately transport-neutral. It carries source facts,
provenance, and coordinate conventions without coupling the library to a
store, a dashboard, or a scientific backend.

In [ ]:
#| default_exp models

In [ ]:
#| export
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any, Mapping

In [ ]:
#| export
@dataclass(frozen=True)
class Provenance:
    source_id: str
    dataset_id: str
    record_version: str = "1"
    imported_from: str | None = None
    ruleset_id: str | None = None

    def as_dict(self) -> dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
@dataclass(frozen=True)
class GenomicCoordinates:
    """0-based, half-open coordinates; arm coordinates are optional."""

    chrom: str
    start: int
    end: int
    strand: str
    rbd5_start: int | None = None
    rbd5_end: int | None = None
    rbd3_start: int | None = None
    rbd3_end: int | None = None

    def __post_init__(self) -> None:
        if self.start < 0 or self.end <= self.start:
            raise ValueError("Coordinates require 0 <= start < end")
        if self.strand not in {"+", "-"}:
            raise ValueError("strand must be '+' or '-'")

    def as_dict(self) -> dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
@dataclass(frozen=True)
class Candidate:
    candidate_id: str
    rbd5: str
    rbd3: str
    provenance: Provenance
    gene_id: str | None = None
    transcript_id: str | None = None
    junction_id: str | None = None
    ligation_site: int | None = None
    shift_nt: int | None = None
    coordinates: GenomicCoordinates | None = None
    on_junction: bool | None = None
    junction_spanning: bool | None = None
    stored_raw_design_score: float | None = None
    extra: Mapping[str, Any] = field(default_factory=dict)

    def __post_init__(self) -> None:
        for arm_name, arm in (("rbd5", self.rbd5), ("rbd3", self.rbd3)):
            if not arm or any(base not in "ACGT" for base in arm.upper()):
                raise ValueError(f"{arm_name} must be a non-empty A/C/G/T sequence")
        if not self.candidate_id:
            raise ValueError("candidate_id is required")

    @property
    def binder(self) -> str:
        return self.rbd5.upper() + self.rbd3.upper()

    @property
    def arm_length(self) -> int:
        if len(self.rbd5) != len(self.rbd3):
            raise ValueError("arm_length is undefined when arms have different lengths")
        return len(self.rbd5)

    @property
    def lig_dinuc(self) -> str:
        return self.rbd5[-1].upper() + self.rbd3[0].upper()

    def as_dict(self) -> dict[str, Any]:
        result = asdict(self)
        result["binder"] = self.binder
        result["lig_dinuc"] = self.lig_dinuc
        return result

In [ ]:
#| export
@dataclass(frozen=True)
class Characterization:
    candidate_id: str
    tm_5p: float
    tm_3p: float
    tm_full: float
    gc_percent: float
    p3_hairpin_dg: float | None
    p3_homodimer_dg: float | None
    p3_heterodimer_dg: float | None
    has_simple_repeat: bool
    has_5g_repeat: bool
    parameters_id: str

    def as_dict(self) -> dict[str, Any]:
        return asdict(self)